# Bateria de Testes: MOEA/D vs NSGA-II (DTLZ1 - 10 Objetivos)

Neste notebook, executamos uma bateria de testes com diferentes orçamentos de avaliações (250, 500, 1000, 1500, 10000).

In [24]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Adiciona o diretório raiz do projeto ao path
sys.path.append(os.path.abspath('.'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.plotting import parallel_coordinates

from src.problems.DTLZ import DTLZ1
from src.MOEAs.MOEAD import MOEAD
from src.MOEAs.NSGAII import NSGAII
from src.MOEAs.crossovers.SBXCrossover import SBXCrossover
from src.MOEAs.mutations.PolynomialMutation import PolynomialMutation
from src.BinaryTournament import BinaryTournament
from src.MOEAs.sparsities.CrowdingDistance import CrowdingDistance
from src.QualityIndicator import HV, IGD
from pymoo.factory import get_reference_directions

# Configuração Base do Problema
M = 10
reference_point = [5.0] * M
hv_calculator = HV(referencePoint=reference_point)
max_volume = 5.0 ** M

# Carregando a Fronteira Verdadeira (Reference Front) para cálculo do IGD
ref_front_path = os.path.join("resources", "ReferenceFronts", "DTLZ", "DTLZ1-pareto_10.txt")
true_front = np.loadtxt(ref_front_path, dtype=float)
igd_calculator = IGD(true_front.tolist())

CASES = [
    {"evaluations": 250, "n_partitions": 2}, 
    {"evaluations": 500, "n_partitions": 2}, 
    {"evaluations": 1000, "n_partitions": 2},
    {"evaluations": 1500, "n_partitions": 2},
    {"evaluations": 10000, "n_partitions": 3},
]

summary_rows = []

for case in CASES:
    evals = case["evaluations"]
    n_part = case["n_partitions"]
    
    # Para M=10: n_partitions=2 gera 55 direções; n_partitions=3 gera 220 direções
    ref_dirs = get_reference_directions("das-dennis", M, n_partitions=n_part)
    pop_size = len(ref_dirs)
    
    print(f"Executando {evals} avaliações (População: {pop_size})...")
    
    # Operadores
    crossover = SBXCrossover(distributionIndex=20, crossoverProbability=1.0)
    mutation = PolynomialMutation(mutationProbability=1.0/(M+4), distributionIndex=20)
    selection = BinaryTournament()
    sparsity = CrowdingDistance()
    
    # MOEA/D
    problem = DTLZ1(numberOfObjectives=M)
    moead = MOEAD(problem=problem, maxEvaluations=evals, populationSize=pop_size, 
                  crossover=crossover, mutation=mutation, selection=selection, 
                  sparsity=sparsity, ref_dirs=ref_dirs, n_neighbors=min(20, pop_size), prob_neighbor_mating=0.9)
    moead.execute()
    front_moead = moead.paretoFront.getInstance().front[0]
    obj_moead = np.array([s.objectives for s in front_moead])
    
    hv_moead = hv_calculator.calculate(obj_moead) / max_volume
    igd_moead = igd_calculator.calculate(obj_moead.tolist()) if obj_moead.size > 0 else float("inf")
    
    # NSGA-II
    problem = DTLZ1(numberOfObjectives=M)
    nsga2 = NSGAII(problem=problem, maxEvaluations=evals, populationSize=pop_size, 
                   offSpringPopulationSize=pop_size // 2, crossover=crossover, 
                   mutation=mutation, selection=selection, sparsity=sparsity)
    nsga2.execute()
    front_nsga2 = nsga2.paretoFront.getInstance().front[0]
    obj_nsga2 = np.array([s.objectives for s in front_nsga2])
    
    hv_nsga2 = hv_calculator.calculate(obj_nsga2) / max_volume
    igd_nsga2 = igd_calculator.calculate(obj_nsga2.tolist()) if obj_nsga2.size > 0 else float("inf")
    
    summary_rows.append({
        "Avaliações": evals,
        "População": pop_size,
        "IGD MOEA/D": round(igd_moead, 4),
        "IGD NSGA-II": round(igd_nsga2, 4),
        # "HV MOEA/D": round(hv_moead, 6),
        # "HV NSGA-II": round(hv_nsga2, 6)
    })

df_results = pd.DataFrame(summary_rows)


Executando 250 avaliações (População: 55)...
Executando 500 avaliações (População: 55)...
Executando 1000 avaliações (População: 55)...
Executando 1500 avaliações (População: 55)...
Executando 10000 avaliações (População: 220)...


In [25]:
df_results

,Avaliações,População,IGD MOEA/D,IGD NSGA-II
0,250,55,36.2993,21.5179
1,500,55,25.7480,37.2287
2,1000,55,28.5793,35.3983
3,1500,55,18.5908,33.1976
4,10000,220,12.6265,24.8936
